# 第 10 课 · 数据存储：CSV 与 SQLite

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 前面三课你已经会「抠数据」了。这一课把它们**保存下来**：先整理成字典列表，
> 再写 CSV（给 Excel 看），最后存进 SQLite（给程序查）。本课只引入：records、csv、sqlite3 参数化插入。

In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import csv
import sqlite3

BASE = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫')
DATA_DIR = BASE / 'data'
OUT_DIR = BASE / 'output'
OUT_DIR.mkdir(exist_ok=True)        # 确保 output 文件夹存在（已存在也不报错）
print('输出目录：', OUT_DIR)

输出目录： C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\output


## 10.1 先把解析结果整理成 records（字典列表）

**生活化类比**：你从每页书上抄下的卡片，先统一写成「字段→值」的小卡片（Python 字典），
攒成一沓（列表）。后面不管写 CSV 还是数据库，喂的都是同一沓卡片。

**是什么**：`records = [{'title':..., 'price':..., 'stars':...}, ...]`。

**为什么需要**：把「解析」和「存储」分开——解析只负责吐字典列表，存储只负责吃字典列表，谁也不关心对方细节。

In [2]:
def parse_books(html_text):
    soup = BeautifulSoup(html_text, 'lxml')
    records = []
    for pod in soup.select('article.product_pod'):
        a = pod.select_one('h3 a')
        records.append({
            'title': a.get('title'),                              # 书名
            'price': pod.select_one('.price_color').get_text(),     # 价格
            'stars': pod.select_one('.star-rating').get('class')[1],# 星级
        })
    return records

records = []
for fname in ['books_index.html', 'books_page2.html']:   # 两页都解析
    text = (DATA_DIR / fname).read_text(encoding='utf-8')
    records.extend(parse_books(text))                    # 把这页的结果追加进 records

print('一共解析到几条：', len(records))
for r in records:
    print(r)

一共解析到几条： 7
{'title': 'A Light in the Attic', 'price': '¥51.77', 'stars': 'Three'}
{'title': 'Tipping the Velvet', 'price': '¥53.74', 'stars': 'One'}
{'title': 'Soumission', 'price': '¥50.10', 'stars': 'Four'}
{'title': 'Sharp Objects', 'price': '¥47.82', 'stars': 'Four'}
{'title': 'Sapiens: A Brief History of Humankind', 'price': '¥54.23', 'stars': 'Two'}
{'title': 'The Requiem Red', 'price': '¥22.65', 'stars': 'One'}
{'title': 'The Dirty Little Secrets of Getting Your Dream Job', 'price': '¥33.34', 'stars': 'Four'}


## 10.2 写 CSV：给 Excel 看的表格

**生活化类比**：CSV 就是「用逗号分格的记事本」，Excel 能直接打开。
但中文直接写 UTF-8，Excel 打开会乱码——所以加一个 **utf-8-sig**（在文件头放一个「暗号」，Excel 见到就不乱码）。

**是什么**：`csv.DictWriter` 按字典的键自动写表头和每一行。

**为什么需要**：CSV 最轻量，人能直接打开看，也是数据库之间交换数据的通用格式。

In [3]:
csv_path = OUT_DIR / 'lesson10_books.csv'
with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:  # utf-8-sig 防 Excel 乱码
    writer = csv.DictWriter(f, fieldnames=['title', 'price', 'stars'])
    writer.writeheader()                 # 第一行写表头 title,price,stars
    writer.writerows(records)            # 把所有字典一次写成多行

print('已写出：', csv_path)
print('回读验证：')
print(csv_path.read_text(encoding='utf-8-sig'))

已写出： C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\output\lesson10_books.csv
回读验证：
title,price,stars
A Light in the Attic,¥51.77,Three
Tipping the Velvet,¥53.74,One
Soumission,¥50.10,Four
Sharp Objects,¥47.82,Four
Sapiens: A Brief History of Humankind,¥54.23,Two
The Requiem Red,¥22.65,One
The Dirty Little Secrets of Getting Your Dream Job,¥33.34,Four



## 10.3 建 SQLite 表，用参数化插入（绝不 f-string 拼 SQL）

**生活化类比**：CSV 是个本子；SQLite 是一个**单文件的迷你数据库**，能按条件查、能自动去重。

**是什么 / 铁律**：
- `sqlite3.connect(文件路径)` 连库（文件不存在就新建）；
- 建表后用 `?` 占位符插入：`INSERT INTO ... VALUES (?, ?, ?)`，值放在**第二个参数的元组**里；
- **绝对禁止**用 f-string 把书名拼进 SQL——会出错，更会被注入攻击；
- `INSERT OR IGNORE`：靠主键去重，重复插入自动跳过。

**为什么需要**：参数化是工业界标准写法，既是安全红线，也省去了自己处理引号的麻烦。

In [4]:
db_path = OUT_DIR / 'lesson10_books.db'
conn = sqlite3.connect(db_path)     # 连接数据库（文件不存在就新建）
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS books (
    title TEXT PRIMARY KEY,   -- 书名做主键：重复的书会被自动识别
    price TEXT,
    stars  TEXT
)
''')

cur.execute('DELETE FROM books')     # 让本课可重复运行：先清空旧数据，再重新插入

for r in records:
    cur.execute(
        'INSERT OR IGNORE INTO books (title, price, stars) VALUES (?, ?, ?)',
        (r['title'], r['price'], r['stars'])   # 值放元组里，绝不用 f-string 拼 SQL
    )
conn.commit()                        # 真正写盘
print('插入完成')

插入完成


## 10.4 SELECT 验证：插进去了，就该查得出来

In [5]:
cur.execute('SELECT title, price, stars FROM books ORDER BY price')
rows = cur.fetchall()
print('表里现在有', len(rows), '条（按价格排序）：')
for row in rows:
    print(row)

# 故意再插一次第一本，验证 INSERT OR IGNORE 真的去重了
cur.execute(
    'INSERT OR IGNORE INTO books (title, price, stars) VALUES (?, ?, ?)',
    (records[0]['title'], records[0]['price'], records[0]['stars']))
conn.commit()
cur.execute('SELECT COUNT(*) FROM books')
print('重复插入第一本后，条数仍是：', cur.fetchone()[0])   # 应与前面相同，说明去重生效

conn.close()

表里现在有 7 条（按价格排序）：
('The Requiem Red', '¥22.65', 'One')
('The Dirty Little Secrets of Getting Your Dream Job', '¥33.34', 'Four')
('Sharp Objects', '¥47.82', 'Four')
('Soumission', '¥50.10', 'Four')
('A Light in the Attic', '¥51.77', 'Three')
('Tipping the Velvet', '¥53.74', 'One')
('Sapiens: A Brief History of Humankind', '¥54.23', 'Two')
重复插入第一本后，条数仍是： 7


## 10.5 随堂小练习（自己动手，空格里写）

In [6]:
# TODO：给 records 增加一列 availability（库存文字），重新解析并重写 CSV

In [7]:
# TODO：用 SELECT 只查出星级为 Four 的书（提示：WHERE stars='Four'）

## 来源与延伸

- Python sqlite3 官方文档：https://docs.python.org/3/library/sqlite3.html
- Python csv 模块文档：https://docs.python.org/zh-cn/3/library/csv.html
- 本课正好衔接「06_SQL」模块：建表、查询、聚合、多表连接，在那个模块系统学。